# Этап 6 V3 — проверка вычислительной реализуемости TabM

## Исследовательский вопрос

Может ли облегчённая конфигурация TabM завершить обучение на KOMUS за разумное время и дать первичную оценку качества?

## Зачем сейчас нужна проверка реализуемости

Этап 6 V1 с `k=32`, тремя внешними фолдами и максимумом 1000 эпох не завершился до получения метрик. Поэтому до повторного полного запуска требуется отдельный диагностический прогон: он отвечает на вопрос о скорости и завершении обучения, а не о превосходстве над GBDT.

## Что проверяем

Запускается только первый фолд заранее зафиксированного `StratifiedKFold`. У TabM уменьшено только `k` до 8 и ограничено число эпох до 100; подбор и поиск параметров отсутствуют. Время измеряется как результат эксперимента и не используется для его остановки.

## Что остаётся неизменным

Используются `Data_final.xlsb`, целевой признак `DefMark`, идентификатор `INN`, те же 47 разрешённых признаков в том же порядке, исключение `Q_B1_norm` и `Q_B2_norm`, проверка SHA-256, предобработка без заполнения пропусков, фиксированное разбиение 80/20 с зерном случайности 42 и зерном 43 для первого фолда. Закрытая 20%-выборка создаётся только как часть воспроизводимого разбиения и немедленно освобождается: она не участвует ни в обучении, ни в прогнозе, ни в метриках.

Этот блокнот не заменяет Этап 6 V1 и не выполняет финальное сравнение моделей. Он сохраняет только результаты V3 в стандартные файлы Stage 6, не затрагивая принятые артефакты других версий.

## Подготовка контракта эксперимента

1. **Что проверяем?** Доступность данных, точную идентичность 47 признаков и параметры облегчённого запуска.
2. **Зачем это делаем сейчас?** Диагностика имеет смысл только при сохранении исходного исследовательского контракта.
3. **Как этот код отвечает на исследовательский вопрос?** Он загружает список признаков из принятого базового результата Этапа 1 и задаёт единственную облегчающую конфигурацию.
4. **Что остаётся неизменным?** Данные, целевой признак, исключённые признаки, порядок признаков и зерна случайности.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import tabm
import torch
import torch.nn.functional as F
from html import escape
from IPython.display import HTML, display
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, TensorDataset


def project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Не найден корень проекта с pyproject.toml.')


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def hash_indices(values: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


def set_fold_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)


ROOT = project_root()
DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
BASELINE_PATH = ROOT / 'reports' / 'generated' / 'stage1_baseline_results_V2.json'
GENERATED_DIR = ROOT / 'reports' / 'generated'
SUMMARY_DIR = ROOT / 'reports' / 'summary'
RESULTS_PATH = GENERATED_DIR / 'stage6_tabm_results_V3.json'
SUMMARY_PATH = SUMMARY_DIR / 'stage6_tabm_summary_V3.json'
OOF_PATH = GENERATED_DIR / 'stage6_tabm_oof_V3.npz'
EXPECTED_DATASET_SHA256 = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_INDEX_SHA256 = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
TARGET = 'DefMark'
IDENTIFIER = 'INN'
FORBIDDEN_FEATURES = ['Q_B1_norm', 'Q_B2_norm']
OUTER_SEED = 42
OUTER_FOLD_NUMBER = 1
FOLD_SEED = 43
MAX_EPOCHS = 100
PATIENCE = 16
TABM_CONFIG = {
    'arch_type': 'tabm', 'k': 8, 'n_blocks': 3, 'd_block': 512,
    'activation': 'ReLU', 'dropout': 0.10, 'start_scaling_init': 'random-signs',
    'num_embeddings': None, 'd_out': 2, 'input_dtype': 'float32',
    'optimizer': 'AdamW', 'lr': 0.002, 'weight_decay': 0.0003,
    'betas': (0.9, 0.999), 'eps': 1e-8, 'gradient_clip_global_norm': 1.0,
    'batch_size': 256, 'share_training_batches': True, 'max_epochs': MAX_EPOCHS,
    'amp': False, 'torch_compile': False, 'scheduler': None, 'warmup': None,
    'class_weights': None, 'sampling': None,
}

with BASELINE_PATH.open(encoding='utf-8') as handle:
    baseline = json.load(handle)
FEATURES = baseline['допустимые_признаки']
if len(FEATURES) != 47 or any(name in FEATURES for name in FORBIDDEN_FEATURES):
    raise ValueError('Нарушен контракт Stage 1: требуется ровно 47 разрешённых признаков.')
BASELINE_XGB_METRICS = baseline['модели']['XGBoost']['итоговые_метрики']
BASELINE_XGB_FOLD_1 = baseline['модели']['XGBoost']['метрики_фолдов'][0]

## Обучение с наблюдаемым прогрессом

1. **Что проверяем?** Способно ли обучение с `k=8` продвигаться по эпохам и завершить отбор эпохи и дообучение.
2. **Зачем это делаем сейчас?** Именно отсутствие наблюдаемого завершения стало ограничением V1.
3. **Как этот код отвечает на исследовательский вопрос?** До и после долгих операций, а также после каждой эпохи обновляется единая панель с этапом, фолдом, числом эпох, минутами, лучшим ROC-AUC и оценкой оставшегося времени.
4. **Что остаётся неизменным?** Архитектурные параметры, кроме `k`, оптимизатор, размер батча, ранняя остановка, способ выбора эпохи и последующее дообучение соответствуют V1.

In [2]:
def elapsed_seconds(started_at: float) -> float:
    return time.perf_counter() - started_at


progress_display = None


def show_progress(
    stage: str,
    epoch: int | None,
    total_epochs: int | None,
    started_at: float,
    best_auc: float | None,
    best_epoch: int | None = None,
    average_epoch_seconds: float | None = None,
    estimated_remaining_seconds: float | None = None,
    note: str | None = None,
) -> None:
    """Обновляет единую понятную панель состояния без строк по батчам."""
    global progress_display
    elapsed = elapsed_seconds(started_at)
    if epoch is not None and total_epochs:
        percent = min(100, round(epoch / total_epochs * 100))
        progress = '█' * round(percent / 5) + '░' * (20 - round(percent / 5))
        epoch_text = f'{epoch} / {total_epochs}'
    else:
        percent, progress, epoch_text = None, '░' * 20, '—'
    auc_text = f'{best_auc:.5f}' if best_auc is not None and np.isfinite(best_auc) else 'пока нет'
    best_epoch_text = str(best_epoch) if best_epoch is not None else 'пока нет'
    average_text = f'{average_epoch_seconds / 60:.2f} мин' if average_epoch_seconds is not None else 'появится после первой эпохи'
    remaining_text = f'{max(0.0, estimated_remaining_seconds) / 60:.1f} мин' if estimated_remaining_seconds is not None else 'пока нельзя оценить'
    note_html = f'<br><b>Примечание:</b> {escape(note)}' if note else ''
    panel = HTML(f"""
    <div style="font-family:Arial; border:1px solid #ccc; padding:12px; border-radius:8px; max-width:660px;">
      <h3 style="margin:0 0 8px 0;">Проверка TabM — текущий статус</h3>
      <b>Этап:</b> {escape(stage)}<br>
      <b>Фолд:</b> {OUTER_FOLD_NUMBER}<br>
      <b>Эпоха:</b> {epoch_text} &nbsp; <b>Прогресс:</b> {progress} {percent if percent is not None else '—'}%<br>
      <b>Лучший ROC-AUC:</b> {auc_text} &nbsp; <b>Лучшая эпоха:</b> {best_epoch_text}<br>
      <b>Прошло времени:</b> {elapsed / 60:.1f} мин &nbsp; <b>Среднее время эпохи:</b> {average_text}<br>
      <b>Осталось примерно:</b> {remaining_text}{note_html}
    </div>
    """)
    if progress_display is None:
        progress_display = display(panel, display_id=True)
    else:
        progress_display.update(panel)


def show_stage(stage: str, started_at: float, best_auc: float | None = None, note: str | None = None) -> None:
    show_progress(stage, None, None, started_at, best_auc, note=note)


def make_model(n_features: int) -> tabm.TabM:
    return tabm.TabM.make(
        n_num_features=n_features, cat_cardinalities=None,
        arch_type=TABM_CONFIG['arch_type'], k=TABM_CONFIG['k'],
        n_blocks=TABM_CONFIG['n_blocks'], d_block=TABM_CONFIG['d_block'],
        activation=TABM_CONFIG['activation'], dropout=TABM_CONFIG['dropout'],
        start_scaling_init=TABM_CONFIG['start_scaling_init'],
        num_embeddings=None, d_out=TABM_CONFIG['d_out'],
    )


def make_loader(x: np.ndarray, y: np.ndarray, seed: int, shuffle: bool) -> DataLoader:
    generator = torch.Generator(device='cpu').manual_seed(seed)
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y.astype(np.int64)))
    return DataLoader(dataset, batch_size=TABM_CONFIG['batch_size'], shuffle=shuffle, generator=generator, num_workers=0, drop_last=False)


def fit_one_epoch(model: tabm.TabM, loader: DataLoader, optimizer: torch.optim.Optimizer, started_at: float, stage: str, epoch: int, total_epochs: int, best_auc: float | None) -> None:
    model.train()
    for x_batch, y_batch in loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(x_batch.float())
        targets = y_batch[:, None].expand(-1, model.k).reshape(-1)
        loss = F.cross_entropy(logits.reshape(-1, 2), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), TABM_CONFIG['gradient_clip_global_norm'])
        optimizer.step()


@torch.inference_mode()
def positive_probabilities(model: tabm.TabM, x: np.ndarray, started_at: float, stage: str, epoch: int, total_epochs: int, best_auc: float | None) -> np.ndarray:
    model.eval()
    tensor = torch.from_numpy(x).float()
    chunks: list[np.ndarray] = []
    starts = list(range(0, len(tensor), TABM_CONFIG['batch_size']))
    for start in starts:
        logits = model(tensor[start:start + TABM_CONFIG['batch_size']])
        chunks.append(torch.softmax(logits, dim=-1).mean(dim=1)[:, 1].cpu().numpy())
    return np.concatenate(chunks)


def make_optimizer(model: tabm.TabM) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=TABM_CONFIG['lr'], weight_decay=TABM_CONFIG['weight_decay'], betas=TABM_CONFIG['betas'], eps=TABM_CONFIG['eps'])


def select_best_epoch(x_outer: np.ndarray, y_outer: np.ndarray, started_at: float) -> tuple[int, float]:
    show_stage('выбор числа эпох: подготовка', started_at)
    fit_idx, inner_idx = train_test_split(np.arange(len(y_outer)), test_size=0.10, stratify=y_outer, random_state=FOLD_SEED)
    set_fold_seed(FOLD_SEED)
    model = make_model(x_outer.shape[1])
    optimizer = make_optimizer(model)
    loader = make_loader(x_outer[fit_idx], y_outer[fit_idx], FOLD_SEED, shuffle=True)
    best_epoch, best_auc, stale_epochs = 0, float('-inf'), 0
    epoch_durations: list[float] = []
    show_stage('выбор числа эпох: модель и данные готовы', started_at)
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_started_at = time.perf_counter()
        fit_one_epoch(model, loader, optimizer, started_at, 'выбор числа эпох: обучение', epoch, MAX_EPOCHS, best_auc)
        show_progress('выбор числа эпох: внутренняя валидация', epoch, MAX_EPOCHS, started_at, best_auc, best_epoch, float(np.mean(epoch_durations)) if epoch_durations else None)
        probability = positive_probabilities(model, x_outer[inner_idx], started_at, 'выбор числа эпох: внутренняя валидация', epoch, MAX_EPOCHS, best_auc)
        auc = float(roc_auc_score(y_outer[inner_idx], probability))
        if auc > best_auc:
            best_epoch, best_auc, stale_epochs = epoch, auc, 0
        else:
            stale_epochs += 1
        progress_state['лучший_внутренний_ROC_AUC'] = best_auc
        executed_epochs['выбор_эпохи'] = epoch
        epoch_durations.append(time.perf_counter() - epoch_started_at)
        eta_seconds = float(np.mean(epoch_durations)) * (MAX_EPOCHS - epoch)
        show_progress('выбор числа эпох: итог эпохи', epoch, MAX_EPOCHS, started_at, best_auc, best_epoch, float(np.mean(epoch_durations)), eta_seconds)
        if stale_epochs >= PATIENCE:
            show_progress('выбор числа эпох: ранняя остановка', epoch, MAX_EPOCHS, started_at, best_auc, best_epoch, float(np.mean(epoch_durations)), 0.0)
            break
    del model, optimizer
    show_progress('выбор числа эпох: завершён', executed_epochs['выбор_эпохи'], MAX_EPOCHS, started_at, best_auc, best_epoch, float(np.mean(epoch_durations)), 0.0)
    return best_epoch, best_auc


def refit_and_predict(x_train: np.ndarray, y_train: np.ndarray, x_valid: np.ndarray, best_epoch: int, best_auc: float, started_at: float) -> np.ndarray:
    show_progress('дообучение: подготовка', 0, best_epoch, started_at, best_auc, best_epoch)
    set_fold_seed(FOLD_SEED)
    model = make_model(x_train.shape[1])
    optimizer = make_optimizer(model)
    loader = make_loader(x_train, y_train, FOLD_SEED, shuffle=True)
    refit_durations: list[float] = []
    show_progress('дообучение: модель и данные готовы', 0, best_epoch, started_at, best_auc, best_epoch)
    for epoch in range(1, best_epoch + 1):
        epoch_started_at = time.perf_counter()
        fit_one_epoch(model, loader, optimizer, started_at, 'дообучение', epoch, best_epoch, best_auc)
        executed_epochs['дообучение'] = epoch
        refit_durations.append(time.perf_counter() - epoch_started_at)
        average_seconds = float(np.mean(refit_durations))
        show_progress('дообучение: итог эпохи', epoch, best_epoch, started_at, best_auc, best_epoch, average_seconds, average_seconds * (best_epoch - epoch))
    show_progress('прогноз: начало', best_epoch, best_epoch, started_at, best_auc, best_epoch, float(np.mean(refit_durations)) if refit_durations else None)
    probability = positive_probabilities(model, x_valid, started_at, 'прогноз внешней валидации', best_epoch, best_epoch, best_auc)
    del model, optimizer
    show_progress('прогноз: завершён', best_epoch, best_epoch, started_at, best_auc, best_epoch, float(np.mean(refit_durations)) if refit_durations else None, 0.0)
    return probability


def metrics(y_true: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    predicted = (probability >= 0.5).astype(np.int64)
    auc = float(roc_auc_score(y_true, probability))
    return {
        'ROC-AUC': auc, 'Gini': 2.0 * auc - 1.0, 'PR-AUC': float(average_precision_score(y_true, probability)),
        'Precision': float(precision_score(y_true, predicted, zero_division=0)),
        'Recall': float(recall_score(y_true, predicted, zero_division=0)),
        'F1': float(f1_score(y_true, predicted, zero_division=0)),
    }

## Контролируемый диагностический запуск

1. **Что проверяем?** Фактическое время одного полного внешнего фолда и наличие первичных метрик.
2. **Зачем это делаем сейчас?** Это минимально достаточная нагрузка для решения, оправдан ли повторный полный запуск.
3. **Как этот код отвечает на исследовательский вопрос?** Он проверяет SHA-256, воспроизводит рабочую выборку, берёт только фолд №1, обучает TabM, рассчитывает метрики на его внешней валидации и печатает понятный итог.
4. **Что остаётся неизменным?** Закрытая часть разбиения не используется; нет подбора гиперпараметров и обучения GBDT. Сохраняются только артефакты текущей версии V3.

In [3]:
started_at = time.perf_counter()
run_status = 'running'
error_details: dict[str, str] | None = None
fold_metrics: dict[str, float] | None = None
best_epoch: int | None = None
best_inner_auc: float | None = None
probability: np.ndarray | None = None
prediction_indices: np.ndarray | None = None
executed_epochs = {'выбор_эпохи': 0, 'дообучение': 0}
progress_state = {'лучший_внутренний_ROC_AUC': None}

try:
    show_stage('проверка данных: контрольная сумма — начало', started_at)
    if sha256_file(DATASET) != EXPECTED_DATASET_SHA256:
        raise ValueError('SHA-256 набора данных не совпадает с зафиксированным контрактом.')
    show_stage('проверка данных: контрольная сумма — завершена', started_at)

    show_stage('загрузка данных: начало', started_at)
    data = pd.read_excel(DATASET, engine='pyxlsb')
    show_stage('загрузка данных: завершена', started_at)
    show_stage('проверка схемы и подготовка предикторов: начало', started_at)
    allowed_columns = [column for column in data.columns if column not in [TARGET, IDENTIFIER, *FORBIDDEN_FEATURES]]
    if allowed_columns != FEATURES:
        raise ValueError('Идентичность или порядок разрешённых признаков отличаются от Stage 1.')
    if any(name in FEATURES or name in allowed_columns for name in FORBIDDEN_FEATURES):
        raise ValueError('Запрещённые признаки обнаружены среди предикторов.')

    x_all = data.loc[:, FEATURES].to_numpy(dtype=np.float32)
    y_all = data[TARGET].to_numpy(dtype=np.int64)
    if x_all.shape[1] != 47 or not np.isfinite(x_all).all():
        raise ValueError('Нарушено число признаков либо обнаружены NaN/non-finite значения; imputation не выполняется.')
    show_stage('проверка схемы и подготовка предикторов: завершена', started_at)

    show_stage('воспроизведение рабочего разбиения: начало', started_at)
    all_indices = np.arange(len(data))
    working_indices, holdout_indices = train_test_split(all_indices, test_size=0.20, stratify=y_all, random_state=OUTER_SEED)
    if len(working_indices) != 289614 or len(holdout_indices) != 72404:
        raise ValueError('Размер зафиксированного разбиения не совпадает с контрактом.')
    if hash_indices(working_indices) != EXPECTED_WORKING_INDEX_SHA256:
        raise ValueError('SHA рабочих индексов не совпадает с контрактом Stage 1.')
    x_work, y_work = x_all[working_indices], y_all[working_indices]
    del data, x_all, y_all, all_indices, holdout_indices
    show_stage('воспроизведение рабочего разбиения: завершено', started_at)

    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED)
    train_idx, valid_idx = next(outer_cv.split(x_work, y_work))
    show_stage('первый внешний фолд: индексы готовы', started_at)
    show_stage('подготовка данных: первый внешний фолд готов', started_at, note=f'Строк обучения: {len(train_idx):,}; строк валидации: {len(valid_idx):,}.')

    best_epoch, best_inner_auc = select_best_epoch(x_work[train_idx], y_work[train_idx], started_at)
    probability = refit_and_predict(x_work[train_idx], y_work[train_idx], x_work[valid_idx], best_epoch, best_inner_auc, started_at)
    prediction_indices = working_indices[valid_idx]
    show_progress('расчёт метрик первого фолда: начало', best_epoch, best_epoch, started_at, best_inner_auc, best_epoch)
    fold_metrics = metrics(y_work[valid_idx], probability)
    show_progress('расчёт метрик первого фолда: завершён', best_epoch, best_epoch, started_at, best_inner_auc, best_epoch, estimated_remaining_seconds=0.0)
    run_status = 'completed'

except KeyboardInterrupt:
    run_status = 'interrupted'
    error_details = {'тип': 'KeyboardInterrupt', 'сообщение': 'Запуск прерван пользователем.'}
    show_stage('запуск прерван пользователем', started_at, progress_state['лучший_внутренний_ROC_AUC'])

except Exception as error:
    run_status = 'error'
    error_details = {'тип': type(error).__name__, 'сообщение': str(error), 'трассировка': traceback.format_exc()}
    show_stage('завершение из-за ошибки', started_at, progress_state['лучший_внутренний_ROC_AUC'], str(error))

runtime_seconds = elapsed_seconds(started_at)
observed_best_inner_auc = best_inner_auc if best_inner_auc is not None else progress_state['лучший_внутренний_ROC_AUC']
baseline_fold_metrics = {key: float(BASELINE_XGB_FOLD_1[key]) for key in ('Gini', 'ROC-AUC', 'PR-AUC', 'Precision', 'Recall', 'F1')}
comparison = None if fold_metrics is None else {key: fold_metrics[key] - baseline_fold_metrics[key] for key in baseline_fold_metrics}
decision = 'первичная проверка завершена; полный запуск требует отдельного решения по фактическому времени' if run_status == 'completed' else 'сначала устранить причину остановки или ошибки'
metadata = {
    'experiment': 'Stage 6', 'version': 'V3', 'status': run_status, 'question': 'Выполнима ли облегчённая конфигурация TabM на первом внешнем фолде без изменения исследовательского контракта?', 'dataset_sha256': EXPECTED_DATASET_SHA256, 'target': TARGET, 'identifier': IDENTIFIER,
    'feature_names_in_order': FEATURES, 'feature_identity_sha256': hashlib.sha256('\n'.join(FEATURES).encode()).hexdigest(),
    'working_rows': 289614, 'working_index_sha256': EXPECTED_WORKING_INDEX_SHA256, 'final_test_used': False,
    'tabm_config': TABM_CONFIG, 'outer_cv': {'type': 'StratifiedKFold', 'n_splits': 3, 'shuffle': True, 'random_state': OUTER_SEED, 'executed_folds': [OUTER_FOLD_NUMBER]}, 'fold_seeds': {'1': FOLD_SEED},
    'early_stopping': {'inner_train_fraction': 0.90, 'inner_validation_fraction': 0.10, 'patience': PATIENCE, 'selection_metric': 'ROC-AUC', 'refit_epochs': 'best_epoch'},
    'baseline': {'primary': 'XGBoost', 'fold_1_metrics': baseline_fold_metrics},
    'best_epoch': best_epoch, 'best_inner_roc_auc': observed_best_inner_auc, 'executed_epochs': executed_epochs,
    'fold_metrics': [] if fold_metrics is None else [{'fold': 1, 'seed': FOLD_SEED, 'best_epoch': best_epoch, 'inner_best_roc_auc': best_inner_auc, 'metrics': fold_metrics}],
    'outer_validation_metrics': fold_metrics, 'oof_metrics': None, 'oof_scope': 'Фолд 1 из 3; полные OOF-метрики не рассчитаны.',
    'runtime_seconds': runtime_seconds, 'comparison_vs_stage1_xgboost_fold1': comparison, 'decision': decision, 'recommendation': decision, 'error': error_details,
    'versions': {'python': platform.python_version(), 'os': platform.platform(), 'cpu_count': os.cpu_count(), 'torch': torch.__version__, 'tabm': tabm.__version__, 'numpy': np.__version__, 'torch_num_threads': torch.get_num_threads()},
    'limitations': ['Выполнен только один внешний фолд.', 'Полные OOF-метрики отсутствуют.', 'Случайная CV не доказывает временную стабильность.', 'Порог 0.5 — только диагностический.', 'Закрытая тестовая выборка не использована.'],
}
summary = {'experiment': 'Stage 6', 'version': 'V3', 'status': run_status, 'runtime_seconds': runtime_seconds, 'executed_epochs': executed_epochs, 'best_epoch': best_epoch, 'best_inner_roc_auc': observed_best_inner_auc, 'metrics': fold_metrics, 'comparison_vs_stage1_xgboost_fold1': comparison, 'decision': decision}
show_stage('сохранение артефактов: начало', started_at, observed_best_inner_auc)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
if probability is not None and prediction_indices is not None and fold_metrics is not None:
    np.savez_compressed(OOF_PATH, y_true=y_work[valid_idx], probability=probability, fold=np.full(len(probability), OUTER_FOLD_NUMBER, dtype=np.int8), indices=prediction_indices)
show_stage('сохранение артефактов: завершено', started_at, observed_best_inner_auc)
status_text = {'completed': 'завершён', 'interrupted': 'прерван пользователем', 'error': 'завершён с ошибкой'}[run_status]
metrics_text = 'Метрики не получены.' if fold_metrics is None else '; '.join(f'{key}: {value:.5f}' for key, value in fold_metrics.items())
comparison_text = 'Сравнение недоступно: метрики первого фолда не получены.' if comparison is None else f'Δ Gini относительно XGBoost Этапа 1 на фолде 1: {comparison["Gini"]:+.5f}. Это предварительное, не итоговое сравнение.'
display(HTML(f'<div style="font-family:Arial; border:2px solid #4c78a8; padding:14px; border-radius:8px; max-width:720px;"><h2>ИТОГ ПРОВЕРКИ</h2><b>Статус:</b> {status_text}<br><b>Фактическое время:</b> {runtime_seconds / 60:.2f} мин<br><b>Выполнено эпох:</b> выбор — {executed_epochs["выбор_эпохи"]}, дообучение — {executed_epochs["дообучение"]}, всего — {sum(executed_epochs.values())}<br><b>Лучшая эпоха:</b> {best_epoch if best_epoch is not None else "нет данных"}<br><b>Лучший внутренний ROC-AUC:</b> {observed_best_inner_auc if observed_best_inner_auc is not None else "нет данных"}<br><br><b>Метрики внешней валидации первого фолда:</b> {metrics_text}<br><br><b>Краткое сравнение:</b> {comparison_text}<br><br><b>Решение:</b> {decision}<br><b>Что результат не доказывает:</b> превосходство над базовой моделью, статистическую значимость и временную стабильность.<br><b>Следующий шаг:</b> {"Перед полным экспериментом утвердить бюджет по фактическому времени первого фолда." if run_status == "completed" else "Проверить сохранённый технический результат и устранить причину остановки."}</div>'))

## ФАКТЫ

В блоке «ИТОГ ПРОВЕРКИ» выше явно выводятся: завершилось ли обучение, время в минутах, число выполненных эпох выбора и дообучения, их сумма, лучшая эпоха, лучший внутренний ROC-AUC, наличие метрик и рекомендация. До выполнения здесь нет предзаполненного результата.

## ИНТЕРПРЕТАЦИЯ

Если статус «завершён» и метрики получены, облегчённая конфигурация технически выполнима. Это основание только для планирования следующего запуска с учётом фактически измеренного времени; это не доказательство качества TabM относительно базового результата. Если запуск прерван или произошла ошибка, полный запуск в текущем виде не рекомендуется.

## ОГРАНИЧЕНИЯ

Один фолд не оценивает вариативность по фолдам и не даёт статистической значимости. Случайная CV не доказывает временную стабильность. Метрики при пороге 0.5 являются только диагностическими. Проверка не сравнивает TabM с GBDT и не использует закрытую часть данных.

## СЛЕДУЩИЙ ШАГ

При успешном завершении зафиксировать фактическую скорость, оценить масштабирование на три фолда и отдельно утвердить вычислительный план перед полным Этапом 6 V3. При неуспехе сначала устранить причину остановки или изменить среду выполнения, затем повторить проверку вычислительной реализуемости.